In [0]:
-- Work inside the workspace catalog and the ecom_poc schema,
-- so new tables are created there without typing the full name every time.
USE CATALOG workspace;
USE SCHEMA ecom_poc;

In [0]:
-- A TEMP VIEW is a saved query, not a saved table. Nothing is stored on disk;
-- it just gives the query a name so later cells can reuse it.
CREATE OR REPLACE TEMP VIEW orders_parsed AS

-- Step A ("cleaned"): make a tidy copy of every Bronze field.
-- The new columns end in _c ("cleaned"). The original text columns stay untouched.
WITH cleaned AS (
  SELECT
    b.*,   -- keep every original Bronze column (raw text plus audit columns)

    -- Text fields: trim spaces; turn blank / 'NULL' / 'NONE' into a real NULL;
    -- standardise the case so ' shipped ' and 'SHIPPED' become the same value.
    CASE WHEN upper(trim(order_id))     IN ('','NULL','NONE') THEN NULL ELSE trim(order_id) END          AS order_id_c,
    CASE WHEN upper(trim(customer_id))  IN ('','NULL','NONE') THEN NULL ELSE trim(customer_id) END       AS customer_id_c,
    CASE WHEN upper(trim(category))     IN ('','NULL','NONE') THEN NULL ELSE initcap(trim(category)) END AS category_c,       -- 'home & kitchen' -> 'Home & Kitchen'
    CASE WHEN upper(trim(country))      IN ('','NULL','NONE') THEN NULL ELSE upper(trim(country)) END    AS country_c,        -- 'in ' -> 'IN'
    CASE WHEN upper(trim(order_status)) IN ('','NULL','NONE') THEN NULL ELSE upper(trim(order_status)) END AS order_status_c, -- 'Paid' -> 'PAID'

    -- Type conversion with try_cast / try_to_date:
    -- a good value converts ('2' -> 2); a bad one ('abc', '$19.99') becomes NULL instead of crashing.
    try_cast(trim(quantity)   AS INT)            AS quantity_c,
    try_cast(trim(unit_price) AS DECIMAL(10,2))  AS unit_price_c,
    try_to_date(trim(order_date), 'yyyy-MM-dd')  AS order_date_c   -- impossible dates like 2025-13-45 become NULL
  FROM bronze_orders b
)

-- Step B: keep everything from Step A and add ONE more column, rejection_reason.
SELECT *,

  -- array(...) builds a list with one slot per rule. A slot holds the reason name
  -- if the rule is broken, otherwise NULL (a CASE with no match returns NULL).
  -- filter(... x IS NOT NULL) throws away the NULL slots, keeping only broken rules.
  -- array_join(..., ';') glues the remaining reasons into one text value.
  -- Result: '' means the row is valid; otherwise it lists every rule the row broke.
  array_join(filter(array(

    -- Mandatory fields: must exist after cleaning.
    CASE WHEN order_id_c IS NULL THEN 'MISSING_ORDER_ID' END,
    CASE WHEN customer_id_c IS NULL THEN 'MISSING_CUSTOMER_ID' END,

    -- Category: missing, or present but not one of the 8 known categories.
    CASE WHEN category_c IS NULL THEN 'MISSING_CATEGORY'
         WHEN category_c NOT IN ('Electronics','Clothing','Home & Kitchen','Books','Beauty','Sports & Outdoors','Toys','Grocery') THEN 'INVALID_CATEGORY' END,

    -- Country: missing, or not one of the 10 supported codes.
    CASE WHEN country_c IS NULL THEN 'MISSING_COUNTRY'
         WHEN country_c NOT IN ('US','GB','DE','FR','CA','AU','IN','BD','SG','JP') THEN 'INVALID_COUNTRY' END,

    -- Status: must be one of the allowed statuses (a blank also fails).
    CASE WHEN order_status_c IS NULL OR order_status_c NOT IN ('PLACED','PAID','SHIPPED','DELIVERED','CANCELLED','RETURNED') THEN 'INVALID_ORDER_STATUS' END,

    -- Quantity: first "is it a number at all?", then "is it in a sensible range?".
    CASE WHEN quantity_c IS NULL THEN 'QUANTITY_NOT_NUMERIC'
         WHEN quantity_c < 1 OR quantity_c > 1000 THEN 'QUANTITY_OUT_OF_RANGE' END,

    -- Unit price: same two-step check.
    CASE WHEN unit_price_c IS NULL THEN 'UNIT_PRICE_NOT_NUMERIC'
         WHEN unit_price_c < 0.01 OR unit_price_c > 100000 THEN 'UNIT_PRICE_OUT_OF_RANGE' END,

    -- Order date: first "is it a real date?", then "is it not in the future?".
    CASE WHEN order_date_c IS NULL THEN 'ORDER_DATE_INVALID'
         WHEN order_date_c > current_date() THEN 'ORDER_DATE_FUTURE' END

  ), x -> x IS NOT NULL), ';') AS rejection_reason
FROM cleaned;

In [0]:
-- Save every row that broke at least one rule (rejection_reason is not empty).
-- We store the ORIGINAL Bronze text values, so the evidence is exactly what arrived,
-- plus the audit columns and the reasons. Nothing is silently deleted.
CREATE OR REPLACE TABLE quarantine_orders
COMMENT 'Orders that failed at least one validation rule, with rejection reasons. Raw Bronze values kept.'
AS
SELECT order_id, customer_id, order_date, category, quantity, unit_price, order_status, country,
       _ingested_at, _source_file, _batch_id, _row_hash, rejection_reason
FROM orders_parsed
WHERE rejection_reason <> '';

In [0]:
-- Only valid rows (rejection_reason = '') get ranked. This is "validate first, then deduplicate".
-- PARTITION BY order_id_c : put rows with the same order_id in one group.
-- ORDER BY _row_hash ASC  : inside each group, sort by hash so the order is always the same.
-- ROW_NUMBER()            : number them 1, 2, 3 ... in that order.
-- rn = 1 is the winner (kept); rn > 1 are duplicate losers.
CREATE OR REPLACE TEMP VIEW orders_ranked AS
SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id_c ORDER BY _row_hash ASC) AS rn
FROM orders_parsed
WHERE rejection_reason = '';

In [0]:
-- Bucket 1: the winners (rn = 1), one row per order_id.
-- We save the CLEANED, correctly typed columns (renamed back to their normal names)
-- and keep the audit columns so every row can be traced back to Bronze.
CREATE OR REPLACE TABLE silver_orders
COMMENT 'Typed, validated orders; exactly one row per order_id'
AS
SELECT order_id_c AS order_id, customer_id_c AS customer_id, order_date_c AS order_date,
       category_c AS category, quantity_c AS quantity, unit_price_c AS unit_price,
       order_status_c AS order_status, country_c AS country,
       _ingested_at, _source_file, _batch_id, _row_hash
FROM orders_ranked WHERE rn = 1;

-- Bucket 3: valid rows that lost the keep-one rule (rn > 1).
-- duplicate_rank shows their position (2 = second copy, and so on).
CREATE OR REPLACE TABLE duplicate_orders
COMMENT 'Valid orders that lost the keep-one rule (lowest _row_hash wins per order_id)'
AS
SELECT order_id_c AS order_id, customer_id_c AS customer_id, order_date_c AS order_date,
       category_c AS category, quantity_c AS quantity, unit_price_c AS unit_price,
       order_status_c AS order_status, country_c AS country,
       _ingested_at, _source_file, _batch_id, _row_hash, rn AS duplicate_rank
FROM orders_ranked WHERE rn > 1;

In [0]:
-- Count all four tables, add the three buckets together, and compare to Bronze.
-- The inner SELECT counts each table; the outer SELECT does the arithmetic.
-- reconciles = true means: Bronze = accepted + quarantined + duplicate losers.
SELECT *, accepted + quarantined + duplicate_losers AS total,
       bronze_rows = accepted + quarantined + duplicate_losers AS reconciles
FROM (
  SELECT (SELECT COUNT(*) FROM bronze_orders)     AS bronze_rows,
         (SELECT COUNT(*) FROM silver_orders)     AS accepted,
         (SELECT COUNT(*) FROM quarantine_orders) AS quarantined,
         (SELECT COUNT(*) FROM duplicate_orders)  AS duplicate_losers
);

In [0]:
-- How many quarantined rows per rejection reason (your "row counts by rule" evidence).
SELECT rejection_reason, COUNT(*) AS rows FROM quarantine_orders GROUP BY rejection_reason ORDER BY rows DESC;

-- Sample rows from the quarantine and duplicate tables (your "sample" evidence).
SELECT * FROM quarantine_orders LIMIT 10;
SELECT * FROM duplicate_orders LIMIT 10;